# Forecast Customer ML Trial

Goal:
1. Test whether customer behaviour before launch can predict launch buyers.
2. Calculate real historical buyer ratios for every launch.
3. Use real buyer ratios for forecast calibration.
4. Later train full customer-level ML models for buyer ranking.


In [27]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.ensemble import HistGradientBoostingClassifier

In [28]:
ORDERS_PATH = "data/raw/orders.csv"
LAUNCHES_PATH = "data/raw/launched_product_details.csv"

print(os.path.exists(ORDERS_PATH), ORDERS_PATH)
print(os.path.exists(LAUNCHES_PATH), LAUNCHES_PATH)

True data/raw/orders.csv
True data/raw/launched_product_details.csv


In [29]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = unicodedata.normalize("NFKD", x)
    x = "".join([c for c in x if not unicodedata.combining(c)])
    x = re.sub(r"[^a-z0-9äöüß\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def clean_numeric(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x in ["", "-", "-%", "nan", "None"]:
        return np.nan

    x = x.replace("€", "")
    x = x.replace("%", "")
    x = x.strip()

    if re.match(r"^\d{1,3}(,\d{3})+$", x):
        x = x.replace(",", "")
    else:
        x = x.replace(",", ".")

    x = re.sub(r"[^0-9.\-]", "", x)

    if x in ["", "-", "."]:
        return np.nan

    return float(x)


def normalize_strategy(x):
    if pd.isna(x):
        return "standard"

    x = normalize_text(x)
    x = x.replace("-", "_").replace(" ", "_")

    mapping = {
        "standard": "standard",
        "standart": "standard",
        "co_creation": "co_creation",
        "cocreation": "co_creation",
        "co": "co_creation",
        "limited_edition": "limited_edition",
        "limited": "limited_edition",
    }

    return mapping.get(x, x)

In [30]:
orders = pd.read_csv(ORDERS_PATH, low_memory=False)
launches = pd.read_csv(LAUNCHES_PATH, sep=",", low_memory=False)

orders.columns = orders.columns.str.strip()
launches.columns = launches.columns.str.strip()

print("orders:", orders.shape)
print("launches:", launches.shape)

display(orders.head())
display(launches.head())

orders: (1580824, 21)
launches: (62, 22)


,order_id,customer_nr,sku,price,date,artikel_name,net_revenue,coupon_code,coupon_product,coupon_channel_short,...,product_category,product,coupon_influencer,quantity,flavour,first_order_date,last_order_date,months_since_first_order,customer_status,nr_of_purchase
0,2000052992,e3de57910d5772b19ea25078e21c146013c257c20e0e0c...,BE20001,37.28972,2021-05-26,MOOD 90 Caps DE,31.696262,imke,NaN,INF,...,Well-being,Mood Kapseln,imkesalander,1,NaN,2021-04-29,2026-04-23,1,RETURNING,2
1,2000077537,4d64fcaa8bf78c45e56073f4e85a0c30de0b9cf3ea2e0d...,BE20001,37.28972,2021-08-24,MOOD 90 Caps DE,31.696262,akhatbrain,NaN,INF,...,Well-being,Mood Kapseln,adrienne_koleszar,1,NaN,2021-08-24,2022-04-22,0,NEW,1
2,2000024030,a64bffe32c15c1697adab6794d31ff6c5f390f0280b273...,BE20001,0.00000,2021-02-22,MOOD 90 Caps DE,0.000000,15be,NaN,INF,...,Well-being,Mood Kapseln,lisa_roeckener,1,NaN,2021-02-22,2021-02-22,0,NEW,1
3,2000108887,8248bcb669a79fed3298f794a0c51cfdabea8256f00bec...,BE20001,37.28972,2021-12-23,MOOD 90 Caps DE,31.696262,colleen,NaN,INF,...,Well-being,Mood Kapseln,colleenschneider_,1,NaN,2021-12-23,2021-12-23,0,NEW,1
4,2000058895,fd04dcc92936f30170c5b1784b272bfd7495fb72f3ac49...,BE20001,37.28972,2021-06-14,MOOD 90 Caps DE,31.696262,wastarasagt,NaN,INF,...,Well-being,Mood Kapseln,wastarasagt,1,NaN,2021-06-14,2021-06-14,0,NEW,1


,sku,artikel_name,product,product_need_area,benefit_keywords,flavour,flavour_group,product_form,launch_date,first_order_quantity,...,Product Use Case / What it is about,Target Group,first_week_quantity_target,first_week_quantity,first_week_quantity_target_accuracy,first_6_week_quantity,first_week_nc,first_6_week_nc,first_week_total_c,first_6_week_total_c
0,BE20351,GUT RESTORE DE/FR/IT 60 caps,Gut Restore,gut_health,"microbiome, probiotics, prebiotics, postbiotic...",no flavour,no_flavour,Capsules,2023-12-19,"8,000",...,"Premium 3-in-1 synbiotic (pre-, pro-, and post...",Consumers needing to restore their microbiome ...,2500,1433,57%,4159,611,1943,1352,3919
1,BE20355,DAILY GUT Powder Pomegranate-Hibiscus - 120g P...,Daily Gut Pulver,gut_health,"microbiome, probiotics, gut_brain_axis, mood, ...",Pomegranate Hibiscus,fruity_floral,Drinking powder,2023-08-07,"1,500",...,"Supports gut microbiota, hormonal activity, an...",Adults (18+) and especially women seeking impr...,-,1454,-%,2939,558,1034,1096,1949
2,BE20356,DAILY GUT PRO Powder Green Apple - 120g PET DE...,Daily Gut Pulver,gut_health,"microbiome, probiotics, gut_brain_axis, mood, ...",Apple,fruity_fresh,Drinking powder,2023-07-23,"1,500",...,"Supports gut microbiota, hormonal activity, an...",Adults (18+) and especially women seeking impr...,-,892,-%,1857,266,505,709,1355
3,BE20357,DAILY GUT Powder Peach Ice Tea - 120g PET DE,Daily Gut Pulver,gut_health,"microbiome, probiotics, gut_brain_axis, mood, ...",Peach Ice Tea,refreshing_tea,Drinking powder,2023-08-07,"1,500",...,"Supports gut microbiota, hormonal activity, an...",Adults (18+) and especially women seeking impr...,-,1453,-%,2914,618,1015,1132,1839
4,BE20358,COLLAGEN + WHEY PULVER Cacao 630g DE/EN,Collagen Whey Pulver,beauty_skin_hair,"collagen, protein, skin_glow, beauty, toning, ...",Choco,chocolate,Drinking powder,2023-08-22,"2,000",...,2-in-1 protein and beauty drink (21g Whey / 10...,Sporty people and beauty-conscious influencers.,-,71,-%,457,15,102,67,414


In [31]:
orders = orders.copy()
launches = launches.copy()

orders["date"] = pd.to_datetime(orders["date"], errors="coerce")
orders["first_order_date"] = pd.to_datetime(orders["first_order_date"], errors="coerce")
orders["last_order_date"] = pd.to_datetime(orders["last_order_date"], errors="coerce")

orders["quantity"] = pd.to_numeric(orders["quantity"], errors="coerce")
orders["price"] = pd.to_numeric(orders["price"], errors="coerce")
orders["net_revenue"] = pd.to_numeric(orders["net_revenue"], errors="coerce")

orders = orders[
    orders["date"].notna()
    & orders["customer_nr"].notna()
    & orders["sku"].notna()
    & (orders["quantity"].fillna(0) > 0)
].copy()

orders["product_norm"] = orders["product"].apply(normalize_text)
orders["flavour_norm"] = orders["flavour"].apply(normalize_text)
orders["category_norm"] = orders["product_category"].apply(normalize_text)

launches["launch_date"] = pd.to_datetime(launches["launch_date"], errors="coerce")
launches["uvp"] = launches["uvp"].apply(clean_numeric)
launches["first_order_quantity"] = launches["first_order_quantity"].apply(clean_numeric)
launches["launch_strategy_type"] = launches["launch_strategy_type"].apply(normalize_strategy)

for col in [
    "first_week_quantity",
    "first_6_week_quantity",
    "first_week_nc",
    "first_6_week_nc",
    "first_week_total_c",
    "first_6_week_total_c",
]:
    launches[col] = launches[col].apply(clean_numeric)

launches = launches[launches["launch_date"].notna()].copy()

launches["product_norm"] = launches["product"].apply(normalize_text)
launches["flavour_norm"] = launches["flavour"].apply(normalize_text)
launches["product_form_norm"] = launches["product_form"].apply(normalize_text)
launches["launch_month"] = launches["launch_date"].dt.month

print("orders cleaned:", orders.shape)
print("launches cleaned:", launches.shape)

display(launches[[
    "sku", "product", "flavour", "product_form", "launch_date",
    "launch_strategy_type", "first_week_quantity", "first_6_week_quantity",
    "first_week_nc", "first_6_week_nc"
]].head())

orders cleaned: (1580821, 24)
launches cleaned: (62, 26)


,sku,product,flavour,product_form,launch_date,launch_strategy_type,first_week_quantity,first_6_week_quantity,first_week_nc,first_6_week_nc
0,BE20351,Gut Restore,no flavour,Capsules,2023-12-19,standard,1433.0,4159.0,611.0,1943.0
1,BE20355,Daily Gut Pulver,Pomegranate Hibiscus,Drinking powder,2023-08-07,limited_edition,1454.0,2939.0,558.0,1034.0
2,BE20356,Daily Gut Pulver,Apple,Drinking powder,2023-07-23,standard,892.0,1857.0,266.0,505.0
3,BE20357,Daily Gut Pulver,Peach Ice Tea,Drinking powder,2023-08-07,limited_edition,1453.0,2914.0,618.0,1015.0
4,BE20358,Collagen Whey Pulver,Choco,Drinking powder,2023-08-22,standard,71.0,457.0,15.0,102.0


# Part 1 — Single Launch Proof of Concept

This section tests whether customer-level behaviour before one historical launch can predict who bought the launch product within the first week and first six weeks.

In [32]:
launches_sorted = launches.sort_values("launch_date").reset_index(drop=True)

display(launches_sorted[[
    "sku", "product", "flavour", "product_form", "launch_date",
    "launch_strategy_type", "first_week_quantity", "first_6_week_quantity"
]].head(20))


,sku,product,flavour,product_form,launch_date,launch_strategy_type,first_week_quantity,first_6_week_quantity
0,BE20356,Daily Gut Pulver,Apple,Drinking powder,2023-07-23,standard,892.0,1857.0
1,BE20355,Daily Gut Pulver,Pomegranate Hibiscus,Drinking powder,2023-08-07,limited_edition,1454.0,2939.0
2,BE20357,Daily Gut Pulver,Peach Ice Tea,Drinking powder,2023-08-07,limited_edition,1453.0,2914.0
3,BE20358,Collagen Whey Pulver,Choco,Drinking powder,2023-08-22,standard,71.0,457.0
4,BE20359,Collagen Whey Pulver,Vanilla,Drinking powder,2023-08-22,standard,34.0,199.0
5,BE20364,MCT C8/C10,no flavour,Oils,2023-10-16,standard,30.0,219.0
6,BE20365,Sleep Spray,Peppermint,Sprays,2023-10-23,standard,521.0,1728.0
7,BE20371,Daily Gut Pulver,Cinnamon,Drinking powder,2023-10-31,limited_edition,933.0,1880.0
8,BE20351,Gut Restore,no flavour,Capsules,2023-12-19,standard,1433.0,4159.0
9,BE20374,Daily Fiber Drink,Cherry,Drinking powder,2024-02-05,standard,1317.0,3862.0


In [33]:
launch = launches_sorted.iloc[0]

launch_sku = launch["sku"]
launch_date = launch["launch_date"]
launch_product = launch["product"]
launch_flavour = launch["flavour"]

print("Selected launch:")
print("sku:", launch_sku)
print("product:", launch_product)
print("flavour:", launch_flavour)
print("launch_date:", launch_date)


Selected launch:
sku: BE20356
product: Daily Gut Pulver
flavour: Apple
launch_date: 2023-07-23 00:00:00


In [34]:
def build_customer_features_for_launch(orders, launch):
    launch_sku = launch["sku"]
    launch_date = launch["launch_date"]
    launch_product_norm = launch["product_norm"]
    launch_flavour_norm = launch["flavour_norm"]

    before = orders[orders["date"] < launch_date].copy()

    # Only customers who existed before the launch.
    customers = before["customer_nr"].dropna().unique()

    print("Customers before launch:", len(customers))

    # RFM features
    customer_features = (
        before.groupby("customer_nr")
        .agg(
            last_order_date=("date", "max"),
            first_order_date=("date", "min"),
            order_count=("order_id", "nunique"),
            total_quantity=("quantity", "sum"),
            total_revenue=("net_revenue", "sum"),
            avg_price=("price", "mean"),
            avg_units_per_order=("quantity", "mean"),
        )
        .reset_index()
    )

    customer_features["recency_days"] = (
        launch_date - customer_features["last_order_date"]
    ).dt.days

    customer_features["customer_age_days"] = (
        launch_date - customer_features["first_order_date"]
    ).dt.days

    customer_features["avg_order_value"] = (
        customer_features["total_revenue"] / customer_features["order_count"].replace(0, np.nan)
    )

    # Affinity: same product family
    before["is_same_product_family"] = (
        before["product_norm"] == launch_product_norm
    ).astype(int)

    product_aff = (
        before.groupby("customer_nr")["is_same_product_family"]
        .mean()
        .rename("same_product_affinity")
        .reset_index()
    )

    # Affinity: same flavour
    before["is_same_flavour"] = (
        before["flavour_norm"] == launch_flavour_norm
    ).astype(int)

    flavour_aff = (
        before.groupby("customer_nr")["is_same_flavour"]
        .mean()
        .rename("same_flavour_affinity")
        .reset_index()
    )

    # Coupon behavior
    before["used_coupon"] = before["coupon_code"].notna().astype(int)

    coupon_features = (
        before.groupby("customer_nr")
        .agg(
            coupon_usage_share=("used_coupon", "mean"),
            influencer_order_share=("coupon_influencer", lambda s: s.notna().mean()),
        )
        .reset_index()
    )

    # Previous launch-product behavior approximation:
    # customers who bought any SKU that is in launch table before this launch.
    historical_launch_skus = set(
        launches.loc[launches["launch_date"] < launch_date, "sku"].astype(str)
    )

    before["is_previous_launch_sku"] = before["sku"].astype(str).isin(historical_launch_skus).astype(int)

    launch_aff = (
        before.groupby("customer_nr")["is_previous_launch_sku"]
        .agg(["max", "mean"])
        .rename(columns={
            "max": "previous_launch_buyer_flag",
            "mean": "previous_launch_order_share",
        })
        .reset_index()
    )

    # Labels: bought selected launch SKU in first week / first 6 weeks
    first_week_end = launch_date + pd.Timedelta(days=6)
    first_6w_end = launch_date + pd.Timedelta(days=41)

    after_1w = orders[
        (orders["date"] >= launch_date)
        & (orders["date"] <= first_week_end)
        & (orders["sku"].astype(str) == str(launch_sku))
    ]

    after_6w = orders[
        (orders["date"] >= launch_date)
        & (orders["date"] <= first_6w_end)
        & (orders["sku"].astype(str) == str(launch_sku))
    ]

    buyers_1w = set(after_1w["customer_nr"].dropna().astype(str))
    buyers_6w = set(after_6w["customer_nr"].dropna().astype(str))

    customer_features["customer_nr_str"] = customer_features["customer_nr"].astype(str)

    customer_features["bought_1w"] = customer_features["customer_nr_str"].isin(buyers_1w).astype(int)
    customer_features["bought_6w"] = customer_features["customer_nr_str"].isin(buyers_6w).astype(int)

    customer_features = customer_features.drop(columns=["customer_nr_str"])

    # Merge all feature blocks
    df = customer_features.merge(product_aff, on="customer_nr", how="left")
    df = df.merge(flavour_aff, on="customer_nr", how="left")
    df = df.merge(coupon_features, on="customer_nr", how="left")
    df = df.merge(launch_aff, on="customer_nr", how="left")

    # Launch context features
    df["launch_sku"] = launch_sku
    df["launch_month"] = int(launch["launch_month"])
    df["launch_uvp"] = float(launch["uvp"]) if pd.notna(launch["uvp"]) else np.nan
    df["launch_strategy_type"] = launch["launch_strategy_type"]

    # Encode strategy manually for first test
    df["is_co_creation"] = (df["launch_strategy_type"] == "co_creation").astype(int)
    df["is_limited_edition"] = (df["launch_strategy_type"] == "limited_edition").astype(int)

    # Fill missing numeric values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(0)

    return df

In [35]:
one_launch_train = build_customer_features_for_launch(orders, launch)

print(one_launch_train.shape)
display(one_launch_train.head())

print("Positive rate 1w:", one_launch_train["bought_1w"].mean())
print("Positive count 1w:", one_launch_train["bought_1w"].sum())

print("Positive rate 6w:", one_launch_train["bought_6w"].mean())
print("Positive count 6w:", one_launch_train["bought_6w"].sum())

Customers before launch: 140002
(140002, 25)


,customer_nr,last_order_date,first_order_date,order_count,total_quantity,total_revenue,avg_price,avg_units_per_order,recency_days,customer_age_days,...,coupon_usage_share,influencer_order_share,previous_launch_buyer_flag,previous_launch_order_share,launch_sku,launch_month,launch_uvp,launch_strategy_type,is_co_creation,is_limited_edition
0,0000212ede4c061d304dd0c6f7316c4d9638edd6e0529b...,2022-03-01,2021-01-04,2,8,156.892523,23.072430,1.0,509,930,...,1.0,1.0,0,0.0,BE20356,7,44.9,standard,0,0
1,00011f9a3488adfcc9812303eff4f93fda3a57ee5e8b49...,2023-03-12,2023-03-12,1,1,29.355140,32.616822,1.0,133,133,...,1.0,0.0,0,0.0,BE20356,7,44.9,standard,0,0
2,00014eabd6cdd62776ce596434adc3b79e3ecea33f71af...,2023-05-08,2022-05-11,4,4,106.004673,32.616822,1.0,76,438,...,1.0,1.0,0,0.0,BE20356,7,44.9,standard,0,0
3,000179a2312e2aede190aa86b8301ad9c6f33d09bff59e...,2023-01-15,2023-01-15,1,3,22.897196,7.632399,1.0,189,189,...,0.0,0.0,0,0.0,BE20356,7,44.9,standard,0,0
4,0001a83ecbb356f79ee89e048058f63e8f202d64f5df7b...,2021-12-27,2021-04-05,2,3,63.295728,26.770596,1.5,573,839,...,1.0,1.0,0,0.0,BE20356,7,44.9,standard,0,0


Positive rate 1w: 0.003328523878230311
Positive count 1w: 466
Positive rate 6w: 0.00613562663390523
Positive count 6w: 859


In [36]:
feature_cols = [
    "order_count",
    "total_quantity",
    "total_revenue",
    "avg_price",
    "avg_units_per_order",
    "recency_days",
    "customer_age_days",
    "avg_order_value",
    "same_product_affinity",
    "same_flavour_affinity",
    "coupon_usage_share",
    "influencer_order_share",
    "previous_launch_buyer_flag",
    "previous_launch_order_share",
    "launch_month",
    "launch_uvp",
    "is_co_creation",
    "is_limited_edition",
]

X = one_launch_train[feature_cols].fillna(0)
y = one_launch_train["bought_6w"]

print("y positive rate:", y.mean())
print("y positive count:", y.sum())

if y.nunique() < 2:
    print("Cannot train: only one class in target.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y,
    )

    model = HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=42,
    )

    model.fit(X_train, y_train)

    p = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, p)
    ap = average_precision_score(y_test, p)

    print("AUC:", auc)
    print("Average precision:", ap)

    pred = (p >= 0.5).astype(int)
    print(classification_report(y_test, pred))

y positive rate: 0.00613562663390523
y positive count: 859
AUC: 0.842267605652635
Average precision: 0.03325956667384074
              precision    recall  f1-score   support

           0       0.99      1.00      1.00     34786
           1       0.00      0.00      0.00       215

    accuracy                           0.99     35001
   macro avg       0.50      0.50      0.50     35001
weighted avg       0.99      0.99      0.99     35001



In [37]:
eval_df = pd.DataFrame({
    "y_true": y_test.values,
    "p_buy": p,
})

overall_rate = eval_df["y_true"].mean()

rows = []
for pct in [0.01, 0.02, 0.05, 0.10, 0.20]:
    n = max(1, int(len(eval_df) * pct))
    top = eval_df.sort_values("p_buy", ascending=False).head(n)

    top_rate = top["y_true"].mean()
    captured_buyers = top["y_true"].sum()
    total_buyers = eval_df["y_true"].sum()

    rows.append({
        "top_percent": pct,
        "n_customers": n,
        "buyer_rate_in_top": top_rate,
        "lift_vs_average": top_rate / overall_rate if overall_rate > 0 else np.nan,
        "captured_buyers": int(captured_buyers),
        "total_buyers": int(total_buyers),
        "recall_in_top": captured_buyers / total_buyers if total_buyers > 0 else np.nan,
    })

topk_df = pd.DataFrame(rows)
display(topk_df)

,top_percent,n_customers,buyer_rate_in_top,lift_vs_average,captured_buyers,total_buyers,recall_in_top
0,0.01,350,0.042857,6.976944,15,215,0.069767
1,0.02,700,0.040000,6.511814,28,215,0.130233
2,0.05,1750,0.037714,6.139710,66,215,0.306977
3,0.10,3500,0.031429,5.116425,110,215,0.511628
4,0.20,7000,0.022857,3.721037,160,215,0.744186


In [38]:
expected_buyers = eval_df["p_buy"].sum()
actual_buyers = eval_df["y_true"].sum()

print("Expected buyers from probability sum:", round(expected_buyers, 1))
print("Actual buyers:", int(actual_buyers))
print("Ratio expected / actual:", round(expected_buyers / actual_buyers, 3))

Expected buyers from probability sum: 211.4
Actual buyers: 215
Ratio expected / actual: 0.983


# Part 2 — Historical Buyer Ratio Table

This section calculates the real buyer penetration ratio for every historical launch:
eligible customers before launch, first-week buyers, first-six-week buyers, buyer ratios, NC shares, and units per customer.

In [39]:
def build_historical_buyer_ratio_table(orders, launches):
    rows = []

    for _, launch in launches.iterrows():
        launch_sku = str(launch["sku"])
        launch_date = launch["launch_date"]

        first_week_end = launch_date + pd.Timedelta(days=6)
        first_6w_end = launch_date + pd.Timedelta(days=41)

        # Customers who existed before this launch
        eligible_customers = set(
            orders.loc[
                orders["date"] < launch_date,
                "customer_nr"
            ].dropna().astype(str)
        )

        eligible_count = len(eligible_customers)

        buyers_1w = set(
            orders.loc[
                (orders["date"] >= launch_date)
                & (orders["date"] <= first_week_end)
                & (orders["sku"].astype(str) == launch_sku),
                "customer_nr"
            ].dropna().astype(str)
        )

        buyers_6w = set(
            orders.loc[
                (orders["date"] >= launch_date)
                & (orders["date"] <= first_6w_end)
                & (orders["sku"].astype(str) == launch_sku),
                "customer_nr"
            ].dropna().astype(str)
        )

        buyers_1w_count = len(buyers_1w)
        buyers_6w_count = len(buyers_6w)

        buyer_ratio_1w = buyers_1w_count / eligible_count if eligible_count > 0 else np.nan
        buyer_ratio_6w = buyers_6w_count / eligible_count if eligible_count > 0 else np.nan

        first_week_total_c = launch.get("first_week_total_c", np.nan)
        first_6_week_total_c = launch.get("first_6_week_total_c", np.nan)
        first_week_nc = launch.get("first_week_nc", np.nan)
        first_6_week_nc = launch.get("first_6_week_nc", np.nan)
        first_week_quantity = launch.get("first_week_quantity", np.nan)
        first_6_week_quantity = launch.get("first_6_week_quantity", np.nan)

        nc_share_1w = (
            first_week_nc / first_week_total_c
            if pd.notna(first_week_nc) and pd.notna(first_week_total_c) and first_week_total_c > 0
            else np.nan
        )

        nc_share_6w = (
            first_6_week_nc / first_6_week_total_c
            if pd.notna(first_6_week_nc) and pd.notna(first_6_week_total_c) and first_6_week_total_c > 0
            else np.nan
        )

        units_per_customer_1w = (
            first_week_quantity / first_week_total_c
            if pd.notna(first_week_quantity) and pd.notna(first_week_total_c) and first_week_total_c > 0
            else np.nan
        )

        units_per_customer_6w = (
            first_6_week_quantity / first_6_week_total_c
            if pd.notna(first_6_week_quantity) and pd.notna(first_6_week_total_c) and first_6_week_total_c > 0
            else np.nan
        )

        rows.append({
            "sku": launch_sku,
            "product": launch.get("product", ""),
            "flavour": launch.get("flavour", ""),
            "product_form": launch.get("product_form", ""),
            "launch_strategy_type": launch.get("launch_strategy_type", ""),
            "launch_date": launch_date,
            "launch_month": launch.get("launch_month", np.nan),
            "uvp": launch.get("uvp", np.nan),

            "eligible_customers_before_launch": eligible_count,

            "buyers_1w_existing": buyers_1w_count,
            "buyers_6w_existing": buyers_6w_count,

            "buyer_ratio_1w_existing": buyer_ratio_1w,
            "buyer_ratio_6w_existing": buyer_ratio_6w,

            "first_week_total_c": first_week_total_c,
            "first_6_week_total_c": first_6_week_total_c,
            "first_week_nc": first_week_nc,
            "first_6_week_nc": first_6_week_nc,

            "nc_share_1w": nc_share_1w,
            "nc_share_6w": nc_share_6w,

            "units_per_customer_1w": units_per_customer_1w,
            "units_per_customer_6w": units_per_customer_6w,

            "first_week_quantity": first_week_quantity,
            "first_6_week_quantity": first_6_week_quantity,
        })

    return pd.DataFrame(rows)

In [40]:
launch_ratio_table = build_historical_buyer_ratio_table(orders, launches)

print("launch_ratio_table:", launch_ratio_table.shape)

display(
    launch_ratio_table[
        [
            "sku",
            "product",
            "flavour",
            "launch_date",
            "launch_strategy_type",
            "eligible_customers_before_launch",
            "buyers_1w_existing",
            "buyers_6w_existing",
            "buyer_ratio_1w_existing",
            "buyer_ratio_6w_existing",
            "first_week_total_c",
            "first_6_week_total_c",
            "first_week_nc",
            "first_6_week_nc",
            "nc_share_1w",
            "nc_share_6w",
            "units_per_customer_1w",
            "units_per_customer_6w",
        ]
    ].sort_values("buyer_ratio_6w_existing", ascending=False).head(20)
)

launch_ratio_table: (62, 23)


,sku,product,flavour,launch_date,launch_strategy_type,eligible_customers_before_launch,buyers_1w_existing,buyers_6w_existing,buyer_ratio_1w_existing,buyer_ratio_6w_existing,first_week_total_c,first_6_week_total_c,first_week_nc,first_6_week_nc,nc_share_1w,nc_share_6w,units_per_customer_1w,units_per_customer_6w
0,BE20351,Gut Restore,no flavour,2023-12-19,standard,159318,10,5647,0.000063,0.035445,1352.0,3919.0,611.0,1943.0,0.451923,0.495790,1.059911,1.061240
13,BE20384,Daily Gut Pulver,Strawberry,2024-04-03,standard,182862,4,6409,0.000022,0.035048,4.0,6409.0,0.0,4004.0,0.000000,0.624746,1.000000,1.288188
59,BE20473,Daily Gut Pulver,Strawberry,2026-03-22,standard,329755,4873,8746,0.014778,0.026523,4873.0,8746.0,2100.0,4458.0,0.430946,0.509719,1.420891,1.367139
14,BE20385,Daily Gut Pulver,Lemon,2024-04-03,standard,182862,5,4358,0.000027,0.023832,5.0,4358.0,0.0,2335.0,0.000000,0.535796,1.000000,1.284305
35,BE20418,Gut Shape,no flavour,2024-12-27,standard,238591,3338,5573,0.013990,0.023358,3338.0,5573.0,1069.0,2028.0,0.320252,0.363897,1.757939,1.884263
16,BE20392,Daily Gut Pulver,Berrymix,2024-04-03,standard,182862,6,3620,0.000033,0.019796,6.0,3620.0,0.0,2097.0,0.000000,0.579282,1.000000,1.222099
10,BE20374,Daily Fiber Drink,Cherry,2024-02-05,standard,169906,1024,3152,0.006027,0.018551,1024.0,3152.0,510.0,1822.0,0.498047,0.578046,1.286133,1.225254
58,BE20472,Daily Gut Pulver,Lemon,2026-03-22,standard,329755,3406,5725,0.010329,0.017361,3406.0,5725.0,1440.0,2666.0,0.422783,0.465677,1.449501,1.386376
12,BE20377,Daily Gut Pulver,Creamy Vanilla,2024-02-19,standard,172656,923,2989,0.005346,0.017312,923.0,2989.0,425.0,1695.0,0.460455,0.567079,1.352113,1.320174
21,BE20400,Daily Gut + Collagen Pulver,Vanilla,2024-06-24,co_creation,204037,2086,3405,0.010224,0.016688,2086.0,3405.0,820.0,1445.0,0.393097,0.424376,1.430010,1.526872


In [41]:
summary_cols = [
    "eligible_customers_before_launch",
    "buyers_1w_existing",
    "buyers_6w_existing",
    "buyer_ratio_1w_existing",
    "buyer_ratio_6w_existing",
    "nc_share_1w",
    "nc_share_6w",
    "units_per_customer_1w",
    "units_per_customer_6w",
]

display(
    launch_ratio_table[summary_cols]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
)

,eligible_customers_before_launch,buyers_1w_existing,buyers_6w_existing,buyer_ratio_1w_existing,buyer_ratio_6w_existing,nc_share_1w,nc_share_6w,units_per_customer_1w,units_per_customer_6w
count,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000
mean,232733.725806,852.209677,2058.080645,0.003615,0.009308,0.331994,0.395254,1.295400,1.317180
std,55843.938338,909.903132,1723.453313,0.003409,0.007915,0.126104,0.102879,0.176386,0.158301
min,140002.000000,4.000000,122.000000,0.000022,0.000799,0.000000,0.218090,1.000000,1.061240
10%,153492.200000,32.600000,397.800000,0.000190,0.001426,0.194632,0.277285,1.098976,1.156037
25%,183783.000000,217.000000,818.500000,0.000776,0.002862,0.276247,0.320610,1.165534,1.206835
50%,234311.000000,663.000000,1617.500000,0.002667,0.007205,0.344072,0.379106,1.284851,1.298420
75%,271302.250000,1124.500000,2790.500000,0.005275,0.013159,0.410228,0.456034,1.411586,1.382402
90%,312813.600000,1738.500000,3809.900000,0.007913,0.018432,0.459602,0.542767,1.520159,1.514557
max,329755.000000,4873.000000,8746.000000,0.014778,0.035445,0.577869,0.641841,1.782443,1.884263


In [42]:
monthly_new_customers = (
    orders.dropna(subset=["first_order_date"])
    .assign(first_order_month=lambda d: d["first_order_date"].dt.to_period("M").astype(str))
    .groupby("first_order_month")["customer_nr"]
    .nunique()
    .reset_index(name="new_customers")
)

monthly_new_customers["date"] = pd.to_datetime(monthly_new_customers["first_order_month"] + "-01")

monthly_new_customers = monthly_new_customers.sort_values("date")

display(monthly_new_customers.tail(24))

print("Recent 3M avg new customers:", monthly_new_customers.tail(3)["new_customers"].mean())
print("Historical avg new customers:", monthly_new_customers["new_customers"].mean())

,first_order_month,new_customers,date
91,2024-05,8195,2024-05-01
92,2024-06,7542,2024-06-01
93,2024-07,7040,2024-07-01
94,2024-08,8145,2024-08-01
95,2024-09,4661,2024-09-01
96,2024-10,5190,2024-10-01
97,2024-11,4192,2024-11-01
98,2024-12,3701,2024-12-01
99,2025-01,6162,2025-01-01
100,2025-02,4527,2025-02-01


Recent 3M avg new customers: 5589.0
Historical avg new customers: 2978.939130434783


In [43]:
launch_ratio_table["launch_year_month"] = (
    launch_ratio_table["launch_date"]
    .dt.to_period("M")
    .astype(str)
)

launch_ratio_table = launch_ratio_table.merge(
    monthly_new_customers[["first_order_month", "new_customers"]],
    left_on="launch_year_month",
    right_on="first_order_month",
    how="left"
)

launch_ratio_table = launch_ratio_table.rename(
    columns={"new_customers": "monthly_new_customers_at_launch"}
)

launch_ratio_table["nc_ratio_1w_vs_monthly_nc"] = (
    launch_ratio_table["first_week_nc"]
    / launch_ratio_table["monthly_new_customers_at_launch"].replace(0, np.nan)
)

launch_ratio_table["nc_ratio_6w_vs_monthly_nc"] = (
    launch_ratio_table["first_6_week_nc"]
    / launch_ratio_table["monthly_new_customers_at_launch"].replace(0, np.nan)
)

display(
    launch_ratio_table[
        [
            "sku",
            "product",
            "launch_date",
            "launch_strategy_type",
            "first_week_nc",
            "first_6_week_nc",
            "monthly_new_customers_at_launch",
            "nc_ratio_1w_vs_monthly_nc",
            "nc_ratio_6w_vs_monthly_nc",
            "nc_share_1w",
            "nc_share_6w",
        ]
    ].sort_values("nc_ratio_6w_vs_monthly_nc", ascending=False).head(20)
)

,sku,product,launch_date,launch_strategy_type,first_week_nc,first_6_week_nc,monthly_new_customers_at_launch,nc_ratio_1w_vs_monthly_nc,nc_ratio_6w_vs_monthly_nc,nc_share_1w,nc_share_6w
0,BE20351,Gut Restore,2023-12-19,standard,611.0,1943.0,3037,0.201185,0.639776,0.451923,0.495790
59,BE20473,Daily Gut Pulver,2026-03-22,standard,2100.0,4458.0,7744,0.271178,0.575671,0.430946,0.509719
35,BE20418,Gut Shape,2024-12-27,standard,1069.0,2028.0,3701,0.288841,0.547960,0.320252,0.363897
13,BE20384,Daily Gut Pulver,2024-04-03,standard,0.0,4004.0,8929,0.000000,0.448426,0.000000,0.624746
58,BE20472,Daily Gut Pulver,2026-03-22,standard,1440.0,2666.0,7744,0.185950,0.344267,0.422783,0.465677
43,BE20439,Daily Collagen Pulver,2025-03-23,standard,753.0,1723.0,5181,0.145339,0.332561,0.427598,0.449752
10,BE20374,Daily Fiber Drink,2024-02-05,standard,510.0,1822.0,5900,0.086441,0.308814,0.498047,0.578046
12,BE20377,Daily Gut Pulver,2024-02-19,standard,425.0,1695.0,5900,0.072034,0.287288,0.460455,0.567079
32,BE20415,Daily Gut Pulver,2024-10-14,standard,267.0,1429.0,5190,0.051445,0.275337,0.356475,0.431071
14,BE20385,Daily Gut Pulver,2024-04-03,standard,0.0,2335.0,8929,0.000000,0.261507,0.000000,0.535796


In [44]:
launch_ratio_table["flag_nc_ratio_too_high"] = (
    launch_ratio_table["nc_ratio_6w_vs_monthly_nc"] > 0.5
)

launch_ratio_table["flag_nc_6w_gt_monthly_nc"] = (
    launch_ratio_table["first_6_week_nc"] > launch_ratio_table["monthly_new_customers_at_launch"]
)

launch_ratio_table["flag_nc_gt_total_customers_1w"] = (
    launch_ratio_table["first_week_nc"] > launch_ratio_table["first_week_total_c"]
)

launch_ratio_table["flag_nc_gt_total_customers_6w"] = (
    launch_ratio_table["first_6_week_nc"] > launch_ratio_table["first_6_week_total_c"]
)

display(
    launch_ratio_table[
        launch_ratio_table[
            [
                "flag_nc_ratio_too_high",
                "flag_nc_6w_gt_monthly_nc",
                "flag_nc_gt_total_customers_1w",
                "flag_nc_gt_total_customers_6w",
            ]
        ].any(axis=1)
    ][
        [
            "sku",
            "product",
            "launch_date",
            "launch_strategy_type",
            "first_week_nc",
            "first_week_total_c",
            "first_6_week_nc",
            "first_6_week_total_c",
            "monthly_new_customers_at_launch",
            "nc_ratio_6w_vs_monthly_nc",
            "nc_share_6w",
            "flag_nc_ratio_too_high",
            "flag_nc_6w_gt_monthly_nc",
            "flag_nc_gt_total_customers_1w",
            "flag_nc_gt_total_customers_6w",
        ]
    ].sort_values("nc_ratio_6w_vs_monthly_nc", ascending=False)
)

,sku,product,launch_date,launch_strategy_type,first_week_nc,first_week_total_c,first_6_week_nc,first_6_week_total_c,monthly_new_customers_at_launch,nc_ratio_6w_vs_monthly_nc,nc_share_6w,flag_nc_ratio_too_high,flag_nc_6w_gt_monthly_nc,flag_nc_gt_total_customers_1w,flag_nc_gt_total_customers_6w
0,BE20351,Gut Restore,2023-12-19,standard,611.0,1352.0,1943.0,3919.0,3037,0.639776,0.495790,True,False,False,False
59,BE20473,Daily Gut Pulver,2026-03-22,standard,2100.0,4873.0,4458.0,8746.0,7744,0.575671,0.509719,True,False,False,False
35,BE20418,Gut Shape,2024-12-27,standard,1069.0,3338.0,2028.0,5573.0,3701,0.547960,0.363897,True,False,False,False


In [45]:
launch_ratio_table["buyer_ratio_1w_existing_clipped"] = (
    launch_ratio_table["buyer_ratio_1w_existing"]
    .clip(
        lower=launch_ratio_table["buyer_ratio_1w_existing"].quantile(0.05),
        upper=launch_ratio_table["buyer_ratio_1w_existing"].quantile(0.95),
    )
)

launch_ratio_table["buyer_ratio_6w_existing_clipped"] = (
    launch_ratio_table["buyer_ratio_6w_existing"]
    .clip(
        lower=launch_ratio_table["buyer_ratio_6w_existing"].quantile(0.05),
        upper=launch_ratio_table["buyer_ratio_6w_existing"].quantile(0.95),
    )
)

launch_ratio_table["nc_ratio_1w_vs_monthly_nc_clipped"] = (
    launch_ratio_table["nc_ratio_1w_vs_monthly_nc"]
    .clip(
        lower=launch_ratio_table["nc_ratio_1w_vs_monthly_nc"].quantile(0.05),
        upper=launch_ratio_table["nc_ratio_1w_vs_monthly_nc"].quantile(0.95),
    )
)

launch_ratio_table["nc_ratio_6w_vs_monthly_nc_clipped"] = (
    launch_ratio_table["nc_ratio_6w_vs_monthly_nc"]
    .clip(
        lower=launch_ratio_table["nc_ratio_6w_vs_monthly_nc"].quantile(0.05),
        upper=launch_ratio_table["nc_ratio_6w_vs_monthly_nc"].quantile(0.95),
    )
)

launch_ratio_table["units_per_customer_1w_clipped"] = (
    launch_ratio_table["units_per_customer_1w"]
    .clip(
        lower=launch_ratio_table["units_per_customer_1w"].quantile(0.05),
        upper=launch_ratio_table["units_per_customer_1w"].quantile(0.95),
    )
)

launch_ratio_table["units_per_customer_6w_clipped"] = (
    launch_ratio_table["units_per_customer_6w"]
    .clip(
        lower=launch_ratio_table["units_per_customer_6w"].quantile(0.05),
        upper=launch_ratio_table["units_per_customer_6w"].quantile(0.95),
    )
)

display(
    launch_ratio_table[
        [
            "sku",
            "product",
            "buyer_ratio_6w_existing",
            "buyer_ratio_6w_existing_clipped",
            "nc_ratio_6w_vs_monthly_nc",
            "nc_ratio_6w_vs_monthly_nc_clipped",
            "units_per_customer_6w",
            "units_per_customer_6w_clipped",
        ]
    ].sort_values("nc_ratio_6w_vs_monthly_nc", ascending=False).head(15)
)

,sku,product,buyer_ratio_6w_existing,buyer_ratio_6w_existing_clipped,nc_ratio_6w_vs_monthly_nc,nc_ratio_6w_vs_monthly_nc_clipped,units_per_customer_6w,units_per_customer_6w_clipped
0,BE20351,Gut Restore,0.035445,0.023808,0.639776,0.443218,1.061240,1.137796
59,BE20473,Daily Gut Pulver,0.026523,0.023808,0.575671,0.443218,1.367139,1.367139
35,BE20418,Gut Shape,0.023358,0.023358,0.547960,0.443218,1.884263,1.581673
13,BE20384,Daily Gut Pulver,0.035048,0.023808,0.448426,0.443218,1.288188,1.288188
58,BE20472,Daily Gut Pulver,0.017361,0.017361,0.344267,0.344267,1.386376,1.386376
43,BE20439,Daily Collagen Pulver,0.015109,0.015109,0.332561,0.332561,1.493083,1.493083
10,BE20374,Daily Fiber Drink,0.018551,0.018551,0.308814,0.308814,1.225254,1.225254
12,BE20377,Daily Gut Pulver,0.017312,0.017312,0.287288,0.287288,1.320174,1.320174
32,BE20415,Daily Gut Pulver,0.014512,0.014512,0.275337,0.275337,1.296531,1.296531
14,BE20385,Daily Gut Pulver,0.023832,0.023808,0.261507,0.261507,1.284305,1.284305


In [46]:
os.makedirs("artifacts", exist_ok=True)

launch_ratio_table.to_csv(
    "artifacts/launch_ratio_table_v2.csv",
    index=False
)

print("Saved: artifacts/launch_ratio_table_v2.csv")

Saved: artifacts/launch_ratio_table_v2.csv


In [47]:
import pickle
import pandas as pd

with open("artifacts/model_artifacts_v2.pkl", "rb") as f:
    artifacts = pickle.load(f)

seg_summary = artifacts["behavioral_segmentation"]["segment_summary"]

display(
    seg_summary.sort_values("avg_monetary", ascending=False)
)

,segment_key,customer_count,avg_recency_days,avg_frequency,avg_monetary,avg_sale_share,avg_launch_purchase_count_24m,avg_unique_launch_skus_24m,avg_launch_share_24m,avg_unique_product_count_24m,avg_unique_category_count_24m,avg_unique_flavour_count_24m,avg_product_diversity_ratio_24m,avg_flavour_diversity_ratio_24m,avg_limited_edition_purchase_count_24m,avg_co_creation_purchase_count_24m,global_share,segment_label,segment_description
2,SEG_2,3919,92.558561,15.340903,1369.532158,0.292216,18.970911,9.912988,0.640290,9.682317,3.892575,9.579995,0.335285,0.344692,2.148762,4.865782,0.011551,Loyal high-value buyers,High order frequency and high total spend.
4,SEG_4,25037,198.433758,6.384910,487.921230,0.278719,6.676079,4.438311,0.652290,4.632304,2.548309,4.566601,0.468958,0.479694,0.614650,1.144906,0.073796,Sale-sensitive buyers,Purchases are concentrated in sale or campaign...
0,SEG_0,68801,341.602099,2.183006,130.777761,0.217730,1.294705,1.063487,0.356517,2.160114,1.743463,1.237962,0.757769,0.387413,0.042499,0.087804,0.202788,Variety seekers,"Customers who try a wider mix of products, cat..."
3,SEG_3,87962,329.842728,1.473602,81.849367,0.205565,1.573884,1.440338,0.907769,1.303279,1.092222,1.410836,0.833445,0.838219,0.121996,0.249426,0.259265,Launch adopters,Above-average tendency to buy newly launched S...
1,SEG_1,153556,1282.701900,1.598720,76.896900,0.019984,0.000300,0.000143,0.000068,0.009508,0.009488,0.000436,0.004120,0.000104,0.000000,0.000000,0.452600,Dormant low-value customers,"Old last purchase dates, low frequency, and lo..."


In [48]:
with open("artifacts/model_artifacts_v2.pkl", "rb") as f:
    artifacts = pickle.load(f)

seg_summary = artifacts["behavioral_segmentation"]["segment_summary"]

display(
    seg_summary[
        [
            "segment_key",
            "segment_label",
            "segment_description",
            "customer_count",
            "global_share",
            "avg_recency_days",
            "avg_frequency",
            "avg_monetary",
            "avg_sale_share",
            "avg_launch_purchase_count_24m",
            "avg_unique_launch_skus_24m",
            "avg_launch_share_24m",
            "avg_unique_product_count_24m",
            "avg_unique_flavour_count_24m",
            "avg_product_diversity_ratio_24m",
        ]
    ].sort_values("avg_monetary", ascending=False)
)

,segment_key,segment_label,segment_description,customer_count,global_share,avg_recency_days,avg_frequency,avg_monetary,avg_sale_share,avg_launch_purchase_count_24m,avg_unique_launch_skus_24m,avg_launch_share_24m,avg_unique_product_count_24m,avg_unique_flavour_count_24m,avg_product_diversity_ratio_24m
2,SEG_2,Loyal high-value buyers,High order frequency and high total spend.,3919,0.011551,92.558561,15.340903,1369.532158,0.292216,18.970911,9.912988,0.640290,9.682317,9.579995,0.335285
4,SEG_4,Sale-sensitive buyers,Purchases are concentrated in sale or campaign...,25037,0.073796,198.433758,6.384910,487.921230,0.278719,6.676079,4.438311,0.652290,4.632304,4.566601,0.468958
0,SEG_0,Variety seekers,"Customers who try a wider mix of products, cat...",68801,0.202788,341.602099,2.183006,130.777761,0.217730,1.294705,1.063487,0.356517,2.160114,1.237962,0.757769
3,SEG_3,Launch adopters,Above-average tendency to buy newly launched S...,87962,0.259265,329.842728,1.473602,81.849367,0.205565,1.573884,1.440338,0.907769,1.303279,1.410836,0.833445
1,SEG_1,Dormant low-value customers,"Old last purchase dates, low frequency, and lo...",153556,0.452600,1282.701900,1.598720,76.896900,0.019984,0.000300,0.000143,0.000068,0.009508,0.000436,0.004120


In [49]:
from itertools import product
import numpy as np
import pandas as pd

score_cols = [
    "need_area_score",
    "benefit_score",
    "product_form_score",
    "flavour_group_score",
    "flavour_score",
    "strategy_score",
    "product_score",
    "price_score",
    "month_score",
]

TARGET_METRICS = [
    "first_week_quantity",
    "first_6_week_quantity",
    "first_week_nc",
    "first_6_week_nc",
    "first_week_total_c",
    "first_6_week_total_c",
]


In [50]:
import pickle

ARTIFACT_PATH = "artifacts/model_artifacts_v2.pkl"

with open(ARTIFACT_PATH, "rb") as f:
    artifacts = pickle.load(f)

launches = artifacts["data"]["launches"].copy()
launch_ratio_table = artifacts["data"].get("launch_ratio_table", pd.DataFrame()).copy()

launches["sku"] = launches["sku"].astype(str)
launch_ratio_table["sku"] = launch_ratio_table["sku"].astype(str)

print("launches:", launches.shape)
print("launch_ratio_table:", launch_ratio_table.shape)
print("launch date range:", launches["launch_date"].min(), "→", launches["launch_date"].max())


launches: (62, 46)
launch_ratio_table: (62, 31)
launch date range: 2023-07-23 00:00:00 → 2026-03-22 00:00:00


In [51]:
def smape(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    return np.mean(2 * np.abs(pred - actual) / (np.abs(actual) + np.abs(pred) + 1e-9))


def weighted_metric_prediction(similar_df, metric, similarity_col="similarity_score"):
    valid = similar_df[[metric, similarity_col]].dropna()
    valid = valid[valid[similarity_col] > 0]
    if valid.empty:
        return np.nan
    return np.average(valid[metric], weights=valid[similarity_col])


def evaluate_weight_set(weights, top_n=7, min_history=3):
    rows = []

    launches_sorted = launches.sort_values("launch_date").copy()

    for _, test_row in launches_sorted.iterrows():
        test_date = pd.to_datetime(test_row["launch_date"])
        candidates = launches_sorted[
            (launches_sorted["launch_date"] < test_date)
            & (launches_sorted["sku"].astype(str) != str(test_row["sku"]))
        ].copy()

        if len(candidates) < min_history:
            continue

        scored = []

        for _, cand in candidates.iterrows():
            scores = {
                "need_area_score": float(test_row["product_need_area_norm"] == cand["product_need_area_norm"]),
                "benefit_score": keyword_overlap_score(test_row["benefit_keywords_norm"], cand["benefit_keywords_norm"]),
                "product_form_score": string_similarity(test_row["product_form_norm"], cand["product_form_norm"]),
                "flavour_group_score": float(test_row["flavour_group_norm"] == cand["flavour_group_norm"]),
                "flavour_score": string_similarity(test_row["flavour_norm"], cand["flavour_norm"]),
                "strategy_score": strategy_similarity(test_row["launch_strategy_type"], cand["launch_strategy_type"]),
                "product_score": string_similarity(test_row["product_norm"], cand["product_norm"]),
                "price_score": price_similarity(test_row["uvp"], cand["uvp"]),
                "month_score": month_circular_similarity(test_row["launch_month"], cand["launch_month"]),
            }

            similarity = sum(weights[col] * scores[col] for col in score_cols)
            cand_dict = cand.to_dict()
            cand_dict["similarity_score"] = similarity
            scored.append(cand_dict)

        similar_df = pd.DataFrame(scored).sort_values("similarity_score", ascending=False).head(top_n)

        result = {
            "test_sku": test_row["sku"],
            "test_product": test_row["product"],
            "test_launch_date": test_date,
        }

        for metric in TARGET_METRICS:
            actual = test_row[metric]
            pred = weighted_metric_prediction(similar_df, metric)
            result[f"{metric}_actual"] = actual
            result[f"{metric}_pred"] = pred
            result[f"{metric}_smape"] = smape([actual], [pred]) if pd.notna(pred) else np.nan

        rows.append(result)

    result_df = pd.DataFrame(rows)
    smape_cols = [f"{m}_smape" for m in TARGET_METRICS]

    return {
        "weights": weights,
        "avg_smape": result_df[smape_cols].mean().mean(),
        "metric_smape": result_df[smape_cols].mean().to_dict(),
        "n_tests": len(result_df),
        "details": result_df,
    }

In [52]:
import re
import unicodedata
from difflib import SequenceMatcher

def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[^a-z0-9äöüß\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def normalize_strategy(x):
    if pd.isna(x) or not str(x).strip():
        return "standard"
    x = normalize_text(x).replace("-", "_").replace(" ", "_")
    mapping = {
        "standard": "standard",
        "standart": "standard",
        "co_creation": "co_creation",
        "cocreation": "co_creation",
        "co": "co_creation",
        "limited_edition": "limited_edition",
        "limited": "limited_edition",
    }
    return mapping.get(x, x or "standard")


def normalize_keyword_list(x):
    if x is None or (not isinstance(x, list) and pd.isna(x)):
        return []
    raw_items = x if isinstance(x, list) else str(x).split(",")
    return sorted({normalize_text(item).replace(" ", "_") for item in raw_items if str(item).strip()})


def keyword_overlap_score(a, b):
    a_set = set(normalize_keyword_list(a))
    b_set = set(normalize_keyword_list(b))
    if not a_set or not b_set:
        return 0.0
    return len(a_set & b_set) / len(a_set | b_set)


def string_similarity(a, b):
    a = normalize_text(a)
    b = normalize_text(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def month_circular_similarity(m1, m2):
    if pd.isna(m1) or pd.isna(m2):
        return 0.0
    distance = abs(int(m1) - int(m2))
    distance = min(distance, 12 - distance)
    return 1 - distance / 6


def price_similarity(p1, p2):
    if pd.isna(p1) or pd.isna(p2) or p1 <= 0 or p2 <= 0:
        return 0.5
    return float(min(p1, p2) / max(p1, p2))


def strategy_similarity(s1, s2):
    s1 = normalize_strategy(s1)
    s2 = normalize_strategy(s2)
    if s1 == s2:
        return 1.0
    pair = {s1, s2}
    if pair == {"co_creation", "limited_edition"}:
        return 0.50
    if pair == {"standard", "limited_edition"}:
        return 0.35
    if pair == {"standard", "co_creation"}:
        return 0.30
    return 0.25

In [53]:
baseline_weights = {
    "need_area_score": 0.30,
    "benefit_score": 0.25,
    "product_form_score": 0.12,
    "flavour_group_score": 0.10,
    "flavour_score": 0.08,
    "strategy_score": 0.07,
    "product_score": 0.04,
    "price_score": 0.02,
    "month_score": 0.02,
}

result = evaluate_weight_set(baseline_weights, top_n=7)

print("n_tests:", result["n_tests"])
print("avg_smape:", result["avg_smape"])
pd.Series(result["metric_smape"]).sort_values()

n_tests: 59
avg_smape: 0.8338433080513759


first_6_week_total_c_smape     0.720608
first_6_week_quantity_smape    0.738750
first_6_week_nc_smape          0.851444
first_week_total_c_smape       0.869542
first_week_quantity_smape      0.884607
first_week_nc_smape            0.938109
dtype: float64

In [54]:
rng = np.random.default_rng(42)

search_results = []

for i in range(500):
    raw = rng.dirichlet(np.ones(len(score_cols)))
    weights = dict(zip(score_cols, raw))

    result_i = evaluate_weight_set(weights, top_n=7)
    search_results.append({
        "iteration": i,
        "avg_smape": result_i["avg_smape"],
        "n_tests": result_i["n_tests"],
        **weights,
    })

search_df = pd.DataFrame(search_results).sort_values("avg_smape").reset_index(drop=True)

search_df.head(10)

,iteration,avg_smape,n_tests,need_area_score,benefit_score,product_form_score,flavour_group_score,flavour_score,strategy_score,product_score,price_score,month_score
0,166,0.806325,59,0.182740,0.123736,0.004193,0.035550,0.183200,0.148905,0.046785,0.053567,0.221323
1,56,0.807785,59,0.236094,0.014475,0.029706,0.138206,0.056525,0.092623,0.094715,0.003657,0.334000
2,431,0.810055,59,0.114186,0.086061,0.082926,0.120878,0.015778,0.046436,0.219211,0.007248,0.307276
3,485,0.812624,59,0.204652,0.078250,0.019161,0.140812,0.088877,0.245787,0.026139,0.019659,0.176662
4,241,0.812804,59,0.043166,0.155543,0.168098,0.012169,0.005455,0.165739,0.136308,0.233246,0.080275
5,32,0.813453,59,0.003920,0.208018,0.019744,0.019036,0.084672,0.175493,0.247603,0.107323,0.134190
6,322,0.813898,59,0.186999,0.221318,0.020644,0.003675,0.041880,0.105860,0.342541,0.038943,0.038139
7,103,0.814398,59,0.246968,0.087480,0.029467,0.039673,0.194604,0.111857,0.047785,0.003857,0.238308
8,479,0.814606,59,0.141866,0.143754,0.003393,0.020119,0.077288,0.198403,0.207389,0.043497,0.164292
9,402,0.814827,59,0.078030,0.101035,0.089954,0.016103,0.030647,0.036321,0.615981,0.012797,0.019133


#Similarity Score Testing

In [55]:
best_weights = search_df.iloc[0][score_cols].to_dict()
best_weights

{'need_area_score': 0.18273999303593613,
 'benefit_score': 0.12373591778610876,
 'product_form_score': 0.00419339637483971,
 'flavour_group_score': 0.03555036152226859,
 'flavour_score': 0.183199570182382,
 'strategy_score': 0.14890492834731223,
 'product_score': 0.04678509477687883,
 'price_score': 0.053567430297808805,
 'month_score': 0.22132330767646494}

In [56]:
best_result = evaluate_weight_set(best_weights, top_n=7)

print("avg_smape:", best_result["avg_smape"])
pd.Series(best_result["metric_smape"]).sort_values()


avg_smape: 0.8063245174932083


first_6_week_total_c_smape     0.696649
first_6_week_quantity_smape    0.702474
first_6_week_nc_smape          0.827459
first_week_total_c_smape       0.843795
first_week_quantity_smape      0.852688
first_week_nc_smape            0.914883
dtype: float64

In [57]:
top_n_results = []

for top_n in [3, 5, 7, 10, 15]:
    r = evaluate_weight_set(best_weights, top_n=top_n)
    top_n_results.append({
        "top_n": top_n,
        "avg_smape": r["avg_smape"],
        "n_tests": r["n_tests"],
        **r["metric_smape"],
    })

pd.DataFrame(top_n_results).sort_values("avg_smape")

,top_n,avg_smape,n_tests,first_week_quantity_smape,first_6_week_quantity_smape,first_week_nc_smape,first_6_week_nc_smape,first_week_total_c_smape,first_6_week_total_c_smape
2,7,0.806325,59,0.852688,0.702474,0.914883,0.827459,0.843795,0.696649
3,10,0.816584,59,0.872456,0.705817,0.934008,0.826732,0.867475,0.693013
4,15,0.817897,59,0.875307,0.714930,0.940743,0.823213,0.860750,0.692438
1,5,0.846394,59,0.893894,0.744592,0.955424,0.861175,0.882086,0.741190
0,3,0.854350,59,0.910778,0.746666,0.991786,0.844864,0.894637,0.737372


## Behavioral Segmentation Multiplier Backtest Preparation

This step recreates the behavioral segment multiplier logic used in the forecasting app, but allows testing different clipping ranges.

The goal is to evaluate:

- How much customer-segment affinity should influence forecast volume
- Whether the current cap of `0.75 → 1.35` is optimal
- If a tighter or wider range improves historical forecast accuracy

We compute:

- similarity-weighted segment affinity from comparable launches
- segment purchase propensity
- final multiplier clipped between tested lower / upper bounds

In [58]:
behavioral_segmentation = artifacts.get("behavioral_segmentation", {})
segment_profile = behavioral_segmentation.get("launch_segment_profile", pd.DataFrame()).copy()
segment_summary = behavioral_segmentation.get("segment_summary", pd.DataFrame()).copy()

print("segmentation enabled:", behavioral_segmentation.get("enabled"))
print("segment_profile:", segment_profile.shape)
print("segment_summary:", segment_summary.shape)

segmentation enabled: True
segment_profile: (265, 3)
segment_summary: (5, 19)


In [59]:
def calculate_test_segment_multiplier(similar_df, strategy, lower=0.75, upper=1.35):
    if segment_profile.empty or segment_summary.empty:
        return 1.0

    ref = similar_df[["sku", "similarity_score"]].copy()
    ref["sku"] = ref["sku"].astype(str)
    ref["similarity_score"] = pd.to_numeric(ref["similarity_score"], errors="coerce").fillna(0.0)
    ref = ref[ref["similarity_score"] > 0]

    profile = segment_profile.copy()
    profile["sku"] = profile["sku"].astype(str)

    merged = ref.merge(profile, on="sku", how="left").dropna(subset=["segment_key"])
    if merged.empty:
        return 1.0

    affinity = (
        merged.assign(weighted_share=merged["similarity_score"] * merged["launch_segment_share"])
        .groupby("segment_key", as_index=False)
        .agg(weighted_share=("weighted_share", "sum"))
    )
    affinity["affinity"] = affinity["weighted_share"] / max(affinity["weighted_share"].sum(), 1e-9)

    if strategy == "co_creation":
        propensity_col = "avg_co_creation_purchase_count_24m"
    elif strategy == "limited_edition":
        propensity_col = "avg_limited_edition_purchase_count_24m"
    else:
        propensity_col = "avg_launch_purchase_count_24m"

    if propensity_col not in segment_summary.columns:
        return 1.0

    global_propensity = float(segment_summary[propensity_col].mean())
    if not np.isfinite(global_propensity) or global_propensity <= 0:
        global_propensity = 1.0

    affinity = affinity.merge(
        segment_summary[["segment_key", propensity_col]],
        on="segment_key",
        how="left"
    )
    affinity[propensity_col] = pd.to_numeric(affinity[propensity_col], errors="coerce").fillna(global_propensity)
    affinity["propensity_ratio"] = affinity[propensity_col] / global_propensity

    raw_multiplier = float((affinity["affinity"] * affinity["propensity_ratio"]).sum())
    return float(np.clip(raw_multiplier, lower, upper))

In [60]:
def evaluate_weight_set_with_segment_clip(weights, top_n=7, lower=0.75, upper=1.35, min_history=3):
    rows = []
    launches_sorted = launches.sort_values("launch_date").copy()

    for _, test_row in launches_sorted.iterrows():
        test_date = pd.to_datetime(test_row["launch_date"])

        candidates = launches_sorted[
            (launches_sorted["launch_date"] < test_date)
            & (launches_sorted["sku"].astype(str) != str(test_row["sku"]))
        ].copy()

        if len(candidates) < min_history:
            continue

        scored = []

        for _, cand in candidates.iterrows():
            scores = {
                "need_area_score": float(test_row["product_need_area_norm"] == cand["product_need_area_norm"]),
                "benefit_score": keyword_overlap_score(test_row["benefit_keywords_norm"], cand["benefit_keywords_norm"]),
                "product_form_score": string_similarity(test_row["product_form_norm"], cand["product_form_norm"]),
                "flavour_group_score": float(test_row["flavour_group_norm"] == cand["flavour_group_norm"]),
                "flavour_score": string_similarity(test_row["flavour_norm"], cand["flavour_norm"]),
                "strategy_score": strategy_similarity(test_row["launch_strategy_type"], cand["launch_strategy_type"]),
                "product_score": string_similarity(test_row["product_norm"], cand["product_norm"]),
                "price_score": price_similarity(test_row["uvp"], cand["uvp"]),
                "month_score": month_circular_similarity(test_row["launch_month"], cand["launch_month"]),
            }

            similarity = sum(weights[col] * scores[col] for col in score_cols)
            cand_dict = cand.to_dict()
            cand_dict["similarity_score"] = similarity
            scored.append(cand_dict)

        similar_df = pd.DataFrame(scored).sort_values("similarity_score", ascending=False).head(top_n)
        segment_multiplier = calculate_test_segment_multiplier(
            similar_df,
            strategy=test_row["launch_strategy_type"],
            lower=lower,
            upper=upper,
        )

        result = {
            "test_sku": test_row["sku"],
            "test_product": test_row["product"],
            "test_launch_date": test_date,
            "segment_multiplier": segment_multiplier,
        }

        for metric in TARGET_METRICS:
            actual = test_row[metric]
            base_pred = weighted_metric_prediction(similar_df, metric)
            pred = base_pred * segment_multiplier if pd.notna(base_pred) else np.nan

            result[f"{metric}_actual"] = actual
            result[f"{metric}_pred"] = pred
            result[f"{metric}_smape"] = smape([actual], [pred]) if pd.notna(pred) else np.nan

        rows.append(result)

    result_df = pd.DataFrame(rows)
    smape_cols = [f"{m}_smape" for m in TARGET_METRICS]

    return {
        "lower": lower,
        "upper": upper,
        "avg_smape": result_df[smape_cols].mean().mean(),
        "metric_smape": result_df[smape_cols].mean().to_dict(),
        "avg_segment_multiplier": result_df["segment_multiplier"].mean(),
        "min_segment_multiplier": result_df["segment_multiplier"].min(),
        "max_segment_multiplier": result_df["segment_multiplier"].max(),
        "n_tests": len(result_df),
        "details": result_df,
    }

In [61]:
clip_results = []

for lower in [0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    for upper in [1.10, 1.20, 1.30, 1.35, 1.40, 1.50]:
        if lower < 1.0 < upper:
            r = evaluate_weight_set_with_segment_clip(
                best_weights,
                top_n=7,
                lower=lower,
                upper=upper,
            )
            clip_results.append({
                "lower": lower,
                "upper": upper,
                "avg_smape": r["avg_smape"],
                "avg_segment_multiplier": r["avg_segment_multiplier"],
                "min_segment_multiplier": r["min_segment_multiplier"],
                "max_segment_multiplier": r["max_segment_multiplier"],
                "n_tests": r["n_tests"],
            })

clip_df = pd.DataFrame(clip_results).sort_values("avg_smape").reset_index(drop=True)
clip_df.head(15)

,lower,upper,avg_smape,avg_segment_multiplier,min_segment_multiplier,max_segment_multiplier,n_tests
0,0.90,1.10,0.813120,0.944264,0.90,1.10,59
1,0.85,1.10,0.816417,0.912397,0.85,1.10,59
2,0.90,1.20,0.819162,0.954314,0.90,1.20,59
3,0.90,1.30,0.820686,0.957081,0.90,1.30,59
4,0.90,1.35,0.821137,0.957929,0.90,1.35,59
5,0.80,1.10,0.821539,0.883431,0.80,1.10,59
6,0.90,1.40,0.821563,0.958776,0.90,1.40,59
7,0.90,1.50,0.822348,0.960471,0.90,1.50,59
8,0.85,1.20,0.822459,0.922448,0.85,1.20,59
9,0.85,1.30,0.823983,0.925215,0.85,1.30,59
